# Análisis de la Frontera Eficiente con Datos desde Excel

Este cuaderno está organizado en secciones claramente identificadas para que puedas explicar cada paso en clase. Sigue las instrucciones en las celdas para cargar tus datos, preparar la información y construir la frontera eficiente usando simulaciones de Monte Carlo.

## 1. Preparación del entorno
En esta sección instalamos (si hace falta) y cargamos las bibliotecas necesarias. En Google Colab la mayoría ya está disponible, pero dejamos el código listo por si se ejecuta en otro entorno.

In [ ]:
# ➕ Opcional: descomenta la siguiente línea si necesitas instalar paquetes adicionales en tu entorno
# !pip install pandas numpy matplotlib scipy openpyxl

In [ ]:
# Importamos las bibliotecas fundamentales para el análisis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize
from IPython.display import display, Markdown

## 2. Carga de datos
En esta sección cargamos el archivo de Excel con precios históricos. Se asume que la primera columna contiene fechas y el resto columnas de precios. Puedes subir el archivo directamente a Colab y ajustar la ruta.

In [ ]:
# Ruta al archivo de Excel que contiene las series de precios
# En Colab puedes subir el archivo y usar la ruta proporcionada en la celda de subida
excel_path = "datos_precios.xlsx"  # ✅ Ajusta este nombre a tu archivo real

# Cargamos el archivo. Indicamos que la primera columna corresponde a fechas y debe ser usada como índice
precio_df = pd.read_excel(excel_path, index_col=0, parse_dates=True)

# Mostramos las primeras filas para confirmar la estructura
precio_df.head()

## 3. Exploración y validación de la frecuencia temporal
Esta sección detecta automáticamente la frecuencia de la serie temporal usando las fechas del índice. Además, calculamos métricas descriptivas básicas para entender los datos.

In [ ]:
# Detectamos la frecuencia utilizando el índice de fechas
frecuencia = pd.infer_freq(precio_df.index)

if frecuencia is None:
    mensaje_freq = "No se pudo inferir la frecuencia exacta (posible presencia de datos faltantes o frecuencia irregular)."
else:
    mensaje_freq = f"La frecuencia detectada es: {frecuencia}"

# Creamos una tabla descriptiva de los precios
resumen_precios = precio_df.describe().T

# Mostramos los resultados de la exploración
display(Markdown(f"### Frecuencia de los datos\n{mensaje_freq}"))
display(Markdown("### Resumen estadístico de los precios"))
display(resumen_precios)

## 4. Conversión a rendimientos y factores de anualización
A partir de los precios calculamos los rendimientos porcentuales simples. También determinamos el factor de anualización dependiendo de la frecuencia detectada.

In [ ]:
# Calculamos rendimientos porcentuales simples
rendimientos = precio_df.pct_change().dropna()

# Definimos factores de anualización según la frecuencia
factores_anualizacion = {
    'B': 252, 'C': 252, 'D': 252, 'W': 52, 'M': 12, 'SM': 24, 'BM': 24,
    'Q': 4, 'BQ': 4, 'A': 1, 'BA': 1
}

if frecuencia is None:
    # Si no se detectó frecuencia, asumimos 252 días hábiles al año como aproximación
    factor_anual = 252
    frecuencia_utilizada = "Asumida diaria (252 observaciones por año)"
else:
    clave = frecuencia.split('-')[0]  # En caso de frecuencias con sufijos (por ejemplo, 'BM-JAN')
    factor_anual = factores_anualizacion.get(clave, 252)
    frecuencia_utilizada = f"Detectada {frecuencia} → factor de anualización = {factor_anual}"

media_anual = rendimientos.mean() * factor_anual
cov_anual = rendimientos.cov() * factor_anual

# Mostramos los resultados
texto = f"Se trabajará con un factor de anualización de **{factor_anual}**.\n\nFrecuencia utilizada: {frecuencia_utilizada}."
display(Markdown(texto))
display(Markdown("### Rendimientos anuales esperados (estimados)"))
display(media_anual.to_frame('Media anual esperada'))

## 5. Configuración de la simulación Monte Carlo
Definimos los parámetros para generar una gran cantidad de portafolios aleatorios. Usamos la distribución de Dirichlet para obtener pesos que suman uno y cubran el espacio de forma uniforme.

In [ ]:
# Parámetros para la simulación
n_portafolios = 40000  # Puedes aumentar o disminuir este número según la capacidad de cómputo
riesgo_cero = 0.02     # Tasa libre de riesgo anual (ajusta según tu análisis)

n_activos = len(media_anual)

# Generamos pesos aleatorios usando la distribución de Dirichlet, garantizando suma = 1
pesos_random = np.random.dirichlet(alpha=np.ones(n_activos), size=n_portafolios)

# Calculamos métricas de cada portafolio
rend_port = pesos_random.dot(media_anual.values)
var_port = np.einsum('ij,jk,ik->i', pesos_random, cov_anual.values, pesos_random)
vol_port = np.sqrt(var_port)
sharpe_port = (rend_port - riesgo_cero) / vol_port

# Identificamos el portafolio de Sharpe máximo y de volatilidad mínima
idx_sharpe = np.argmax(sharpe_port)
idx_vol = np.argmin(vol_port)

port_optimo = {
    'Rendimiento': rend_port[idx_sharpe],
    'Volatilidad': vol_port[idx_sharpe],
    'Sharpe': sharpe_port[idx_sharpe],
    'Pesos': pesos_random[idx_sharpe]
}

port_min_vol = {
    'Rendimiento': rend_port[idx_vol],
    'Volatilidad': vol_port[idx_vol],
    'Sharpe': sharpe_port[idx_vol],
    'Pesos': pesos_random[idx_vol]
}

# Presentamos un resumen para la clase
display(Markdown("### Portafolio con mayor ratio de Sharpe (estimado)"))
display(pd.Series(port_optimo, index=['Rendimiento', 'Volatilidad', 'Sharpe']).to_frame('Valor'))
display(Markdown("### Portafolio de mínima volatilidad"))
display(pd.Series(port_min_vol, index=['Rendimiento', 'Volatilidad', 'Sharpe']).to_frame('Valor'))

## 6. Frontera eficiente mediante optimización cuadrática
Para complementar la simulación, resolvemos un problema de optimización para una serie de niveles objetivo de rendimiento. Esto aproxima la frontera eficiente teórica y permite compararla con los puntos aleatorios.

In [ ]:
# Funciones auxiliares para la optimización

def objetivo_volatilidad(pesos, cov):
    # Función objetivo: volatilidad del portafolio
    return np.sqrt(pesos.T @ cov @ pesos)

def restriccion_pesos(pesos):
    return np.sum(pesos) - 1

def restriccion_rendimiento_objetivo(pesos, retorno_objetivo, medias):
    return pesos @ medias - retorno_objetivo

bounds = tuple((0.0, 1.0) for _ in range(n_activos))
restriccion_suma = {'type': 'eq', 'fun': restriccion_pesos}

retornos_objetivo = np.linspace(rend_port.min(), rend_port.max(), 50)
vol_frontera = []

pesos_inicial = np.repeat(1 / n_activos, n_activos)

for retorno_target in retornos_objetivo:
    restricciones = (
        restriccion_suma,
        {'type': 'eq', 'fun': restriccion_rendimiento_objetivo, 'args': (retorno_target, media_anual.values)}
    )

    resultado = minimize(
        objetivo_volatilidad,
        pesos_inicial,
        args=(cov_anual.values,),
        method='SLSQP',
        bounds=bounds,
        constraints=restricciones
    )

    if resultado.success:
        vol_frontera.append(resultado.fun)
    else:
        vol_frontera.append(np.nan)

vol_frontera = np.array(vol_frontera)

## 7. Visualización integral de la frontera eficiente
Finalmente, graficamos los portafolios aleatorios, la frontera eficiente calculada y resaltamos el portafolio con mayor ratio de Sharpe y el de mínima volatilidad.

In [ ]:
plt.figure(figsize=(12, 8))

# Nube de portafolios simulados
scatter = plt.scatter(vol_port, rend_port, c=sharpe_port, cmap='viridis', alpha=0.6, s=12)

# Frontera eficiente calculada
plt.plot(vol_frontera, retornos_objetivo, color='red', linewidth=2.5, label='Frontera eficiente (optimización)')

# Portafolio óptimo y de mínima volatilidad
plt.scatter(port_optimo['Volatilidad'], port_optimo['Rendimiento'], color='orange', s=120, marker='*', label='Mayor Sharpe')
plt.scatter(port_min_vol['Volatilidad'], port_min_vol['Rendimiento'], color='blue', s=120, marker='X', label='Mínima volatilidad')

plt.colorbar(scatter, label='Ratio de Sharpe')
plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.gca().xaxis.set_major_formatter(PercentFormatter(1))
plt.title('Frontera eficiente y portafolios simulados')
plt.xlabel('Volatilidad anualizada')
plt.ylabel('Rendimiento anualizado esperado')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 8. Interpretación y próximos pasos
- **Distribución de los portafolios:** al usar la distribución de Dirichlet, los pesos se distribuyen de forma uniforme sobre el simplex, evitando concentración en una región pequeña de la gráfica.
- **Portafolio óptimo (Sharpe máximo):** es el que ofrece la mejor compensación entre riesgo y retorno respecto a la tasa libre de riesgo definida.
- **Ajustes recomendados:**
  - Cambiar `riesgo_cero` si tienes una estimación más precisa de la tasa libre de riesgo.
  - Ajustar `n_portafolios` para controlar el tiempo de cómputo y densidad de la gráfica.
  - Revisar la frecuencia detectada para confirmar que coincide con la realidad de tus datos.

Este flujo de trabajo sirve como base para extender el análisis a restricciones adicionales (por ejemplo, límites por activo) o para comparar distintos horizontes temporales.